# Representaciones dispersas con Dictionary Learning y K-SVD

**Proyecto 1, Matematicas para la Computacion.**

Este notebook recorre el proceso completo, de la senal cruda al
diccionario entrenado. No reimplementa nada: todo lo importa de los
modulos de `src/`, que son los que se entregan. Aqui solo se explica
y se visualiza.

## El problema en una linea

Dada una senal `x`, encontrar un diccionario `D` y un vector de
coeficientes `alpha` con muy pocas entradas distintas de cero, tales
que `x = D alpha`.

Si `D` tiene 256 columnas y `alpha` solo 9 distintas de cero, una
senal de 128 muestras queda descrita con 9 numeros en vez de 128.

## Mapa del notebook

1. La senal de ECG y como se corta en ventanas
2. Por que el problema tiene infinitas soluciones
3. OMP: encontrar `alpha` con `D` fijo
4. K-SVD: encontrar `D`, y donde entra la descomposicion en valores
   singulares
5. Entrenamiento sobre el ECG real
6. Resultados

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from Datos import cargar_ecg, construir_x, partir_en_ventanas
from Experimento import ARCHIVO_MODELO, curva_error, partir_entrenamiento
from Figuras import elegir_ventana
from Ksvd import actualizar_atomo, diccionario_inicial, ksvd
from Omp import omp, omp_matriz
from Utils import (ERROR_OMP, FRECUENCIA_HZ, ITERACIONES_KSVD,
                   MAX_ATOMOS_OMP, N_ATOMOS, SEMILLA, TAM_VENTANA,
                   dispersion_media, error_relativo,
                   normalizar_columnas)

# La misma paleta validada que usan las figuras del reporte
TINTA = "#52514e"
REJILLA = "#e8e7e2"
AZUL = "#2a78d6"
NARANJA = "#eb6834"
AQUA = "#1baf7a"

PLANTILLA = dict(
    template="plotly_white",
    font=dict(size=12, color=TINTA),
    margin=dict(l=60, r=30, t=60, b=50),
)

print(f"ventana {TAM_VENTANA} muestras | {N_ATOMOS} atomos | "
      f"error objetivo {ERROR_OMP} | tope {MAX_ATOMOS_OMP} atomos")

ventana 128 muestras | 256 atomos | error objetivo 0.1 | tope 16 atomos


## 1. La senal

Se usa el registro 100 de **MIT-BIH Arrhythmia**, de PhysioNet. Son
30 minutos de ECG muestreados a 360 Hz. `wfdb` lo baja del servidor y
lo entrega ya convertido a milivolts.

In [2]:
senal, fs = cargar_ecg()

print(f"{len(senal)} muestras a {fs} Hz = "
      f"{len(senal) / fs / 60:.1f} minutos")

# Diez segundos, que es donde se alcanzan a contar los latidos
tramo = slice(0, 10 * fs)
t = np.arange(len(senal))[tramo] / fs

fig = go.Figure(go.Scatter(x=t, y=senal[tramo], mode="lines",
                           line=dict(color=TINTA, width=1.5),
                           name="ECG"))
fig.update_layout(title="Registro 100, primeros 10 segundos",
                  xaxis_title="segundos", yaxis_title="mV",
                  height=320, showlegend=False, **PLANTILLA)
fig.show()

650000 muestras a 360 Hz = 30.1 minutos


### Por que se corta en ventanas

A K-SVD no se le pasa una senal de 650 mil muestras. Se corta en
ventanas de `TAM_VENTANA = 128` muestras, que a 360 Hz son 356 ms,
poco mas que un latido. Cada ventana es una columna de la matriz `X`.

A cada ventana se le hacen dos cosas antes de guardarla:

- **Se le quita su media.** El ECG trae deriva de linea base, un
  vaiven lento del electrodo. Sin quitarla, el primer atomo que
  aprende el diccionario es una constante, que no describe ninguna
  forma de onda.
- **Se normaliza a norma 1.** Asi lo unico que distingue a una
  columna de otra es la forma de onda, no su amplitud.

In [3]:
crudas = partir_en_ventanas(senal)
x = construir_x(senal)

print(f"X de {x.shape[0]} x {x.shape[1]}")
print(f"normas de columna: min {np.linalg.norm(x, axis=0).min():.4f} "
      f"max {np.linalg.norm(x, axis=0).max():.4f}")

ms = np.arange(TAM_VENTANA) / fs * 1000
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Ventanas crudas",
                                    "Centradas y normalizadas"))

for j in range(0, 40, 4):
    fig.add_trace(go.Scatter(x=ms, y=crudas[:, j], mode="lines",
                             line=dict(color=TINTA, width=1),
                             opacity=0.45, showlegend=False),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=ms, y=x[:, j], mode="lines",
                             line=dict(color=AZUL, width=1),
                             opacity=0.45, showlegend=False),
                  row=1, col=2)

fig.update_xaxes(title_text="ms")
fig.update_layout(height=340,
                  title="Diez ventanas, antes y despues del "
                        "preprocesamiento", **PLANTILLA)
fig.show()

X de 128 x 5078
normas de columna: min 1.0000 max 1.0000


## 2. Por que el problema tiene infinitas soluciones

El diccionario es **sobrecompleto**: tiene mas columnas que filas.
Con `D` de 128 x 256, el sistema

```
x = D alpha
```

tiene 128 ecuaciones y 256 incognitas. Esta subdeterminado, asi que
no tiene una solucion sino un subespacio afin de soluciones de
dimension 128.

Eso no es un defecto, es el diseno. De todas esas soluciones se busca
la **mas dispersa**, la que usa menos atomos. Un diccionario cuadrado
y de rango completo daria una sola solucion, y no habria nada que
elegir.

El problema es que buscar la solucion mas dispersa es NP-dificil. Por
eso se usa un metodo voraz, OMP, que no garantiza el optimo pero en
la practica se le acerca.

In [4]:
d_inicial = diccionario_inicial(x)
print(f"D: {d_inicial.shape[0]} x {d_inicial.shape[1]}")
print(f"ecuaciones: {d_inicial.shape[0]}, "
      f"incognitas: {d_inicial.shape[1]}")
print(f"dimension del espacio de soluciones: "
      f"{d_inicial.shape[1] - np.linalg.matrix_rank(d_inicial)}")

D: 128 x 256
ecuaciones: 128, incognitas: 256
dimension del espacio de soluciones: 129


## 3. OMP: encontrar `alpha` con `D` fijo

Orthogonal Matching Pursuit es voraz. En cada vuelta:

1. Escoge el atomo **mas correlacionado con el residual**, es decir el
   de mayor `|d_i . r|`.
2. Resuelve minimos cuadrados sobre **todos** los atomos escogidos
   hasta el momento, no solo el nuevo.
3. Actualiza el residual.
4. Para cuando el error relativo baja de `ERROR_OMP` o se agota el
   presupuesto `MAX_ATOMOS_OMP`.

El paso 2 es el que le da el nombre: resolver sobre todo el conjunto
deja el residual **ortogonal** al espacio generado por los atomos
elegidos, y por eso ninguno se vuelve a escoger.

### Dos trampas de la notacion de las slides

- `alpha = D' x` **no** da la representacion dispersa. Solo invierte
  `x = D alpha` cuando `D` es ortogonal, y aqui es sobrecompleto a
  proposito. Da las correlaciones, que son el punto de partida del
  paso 1, no la solucion.
- `(D_I)^-1` es una **pseudoinversa**. `D_I` son los atomos
  seleccionados, una matriz alta y rectangular que no tiene inversa.
  Ahi se resuelve un problema de minimos cuadrados.

### OMP vuelta por vuelta

Se toma una ventana con un latido completo y se corre OMP con un tope
creciente de atomos, para ver como va bajando el residual.

In [5]:
col = elegir_ventana(x)
objetivo = x[:, col]

topes = range(1, 13)
errores, reconstrucciones = [], []

for tope in topes:
    coef, elegidos = omp(d_inicial, objetivo, error=0.0,
                         max_atomos=tope)
    recon = d_inicial @ coef
    reconstrucciones.append(recon)
    errores.append(error_relativo(objetivo, recon))

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.55, 0.45],
    subplot_titles=("Reconstruccion acumulada",
                    "Error relativo por atomo agregado"))

fig.add_trace(go.Scatter(x=ms, y=objetivo, mode="lines",
                         name="original",
                         line=dict(color=TINTA, width=3)),
              row=1, col=1)
for tope in (1, 2, 4, 8, 12):
    fig.add_trace(go.Scatter(x=ms, y=reconstrucciones[tope - 1],
                             mode="lines", name=f"{tope} atomos",
                             line=dict(width=1.6)),
                  row=1, col=1)

fig.add_trace(go.Scatter(x=list(topes), y=errores, mode="lines+markers",
                         line=dict(color=AZUL, width=2),
                         marker=dict(size=8), showlegend=False),
              row=1, col=2)
fig.add_hline(y=ERROR_OMP, line=dict(color=NARANJA, dash="dash"),
              annotation_text=f"ERROR_OMP = {ERROR_OMP}",
              annotation_position="top right", row=1, col=2)

fig.update_xaxes(title_text="ms", row=1, col=1)
fig.update_xaxes(title_text="atomos usados", row=1, col=2)
fig.update_yaxes(title_text="error relativo", row=1, col=2)
fig.update_layout(height=400,
                  title="OMP sobre una ventana, con el diccionario "
                        "sin entrenar", **PLANTILLA)
fig.show()

print(f"error con 1 atomo: {errores[0]:.4f}")
print(f"error con 12 atomos: {errores[-1]:.4f}")

error con 1 atomo: 0.2472
error con 12 atomos: 0.0562


### Validacion de OMP

Antes de confiar en OMP hay que probarlo donde **si se sabe la
respuesta**: se arma un diccionario al azar, se generan senales
usando un conjunto conocido de atomos, y se revisa si OMP recupera
justo esos. Si esto falla, cualquier problema de K-SVD es consecuencia
y no causa.

In [6]:
rng = np.random.default_rng(SEMILLA)
k_p, m_p, n_p, usados = 40, 100, 300, 5

d_prueba = normalizar_columnas(rng.standard_normal((k_p, m_p)))

aciertos = exactos = 0
for _ in range(n_p):
    verdaderos = rng.choice(m_p, usados, replace=False)
    sintetica = d_prueba[:, verdaderos] @ rng.standard_normal(usados)
    _, elegidos = omp(d_prueba, sintetica, error=1e-6,
                      max_atomos=usados)
    comunes = len(set(elegidos) & set(verdaderos))
    aciertos += comunes
    exactos += comunes == usados

print(f"diccionario {k_p} x {m_p}, {n_p} senales de {usados} atomos")
print(f"  atomos recuperados:        {aciertos / (n_p * usados):.2%}")
print(f"  senales con soporte exacto: {exactos / n_p:.2%}")

diccionario 40 x 100, 300 senales de 5 atomos
  atomos recuperados:        99.87%
  senales con soporte exacto: 99.33%


## 4. K-SVD: encontrar `D`

K-SVD alterna dos pasos, igual que k-means:

- Con `D` fijo, calcula todas las dispersiones con OMP.
- Con las dispersiones fijas, actualiza `D` **un atomo a la vez**.

### La actualizacion de un atomo, que es donde entra la SVD

Para actualizar el atomo `j`:

1. Se toman solo las senales `w` que **si lo usan**, o sea donde
   `alpha[j, l] != 0`.
2. Se arma el residual `R = X_w - D alpha_w`, con la fila `j` de
   `alpha_w` puesta a cero. Es decir: que queda sin explicar de esas
   senales si se le quita la contribucion del atomo `j`.
3. Se busca el atomo `d_j` y los coeficientes `alpha_j` que mejor
   expliquen `R`.

Y aqui esta el punto. El producto `d_j alpha_j'` es columna por
renglon, o sea una **matriz de rango 1**. Preguntar cual es el mejor
atomo equivale a preguntar cual es la mejor aproximacion de rango 1
de `R`, y eso lo contesta el teorema de **Eckart-Young**:

```
R ~= sigma_1 u_1 v_1'   ->   d_j = u_1 ,   alpha_j = sigma_1 v_1
```

Una SVD por atomo y por iteracion. De ahi el nombre del algoritmo.

**Por que solo las senales de `w`.** Si la actualizacion usara todas
las senales, el atomo entraria en senales que no lo estaban usando y
se destruiria la dispersion que OMP acababa de construir. Restringir
a `w` conserva el patron de ceros.

### Viendo la aproximacion de rango 1

Se toma un atomo concreto, se arma su residual y se compara contra su
mejor aproximacion de rango 1. Si el primer valor singular domina a
los demas, el residual casi es de rango 1 y la actualizacion captura
casi todo.

In [7]:
entrena, prueba = partir_entrenamiento(x)

d_demo = diccionario_inicial(entrena)
alpha_demo = omp_matriz(d_demo, entrena[:, :600])

# Un atomo que use bastantes senales, para que el residual tenga
# suficientes columnas
usos = np.count_nonzero(alpha_demo, axis=1)
j = int(np.argmax(usos))
w = np.flatnonzero(alpha_demo[j, :])

coeficientes = alpha_demo[:, w].copy()
coeficientes[j, :] = 0.0
# w indexa dentro del subconjunto de 600 con el que se calculo
# alpha_demo, asi que el residual se arma sobre ese mismo subconjunto
residual = entrena[:, :600][:, w] - d_demo @ coeficientes

u, sigma, vt = np.linalg.svd(residual, full_matrices=False)
energia = sigma ** 2 / (sigma ** 2).sum()

print(f"atomo {j}, usado por {len(w)} senales")
print(f"residual de {residual.shape[0]} x {residual.shape[1]}")
print(f"energia del primer valor singular: {energia[0]:.1%}")

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                    subplot_titles=("Espectro del residual",
                                    "Atomo viejo contra atomo nuevo"))

fig.add_trace(go.Bar(x=list(range(1, 16)), y=energia[:15],
                     marker_color=AZUL, showlegend=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=ms, y=d_demo[:, j], mode="lines",
                         name="atomo antes",
                         line=dict(color=TINTA, width=2)),
              row=1, col=2)
fig.add_trace(go.Scatter(x=ms, y=u[:, 0], mode="lines",
                         name="u_1, atomo despues",
                         line=dict(color=AZUL, width=2)),
              row=1, col=2)

fig.update_xaxes(title_text="valor singular", row=1, col=1)
fig.update_yaxes(title_text="fraccion de energia", row=1, col=1)
fig.update_xaxes(title_text="ms", row=1, col=2)
fig.update_layout(height=380,
                  title="La actualizacion de un atomo es una "
                        "aproximacion de rango 1", **PLANTILLA)
fig.show()

atomo 50, usado por 130 senales
residual de 128 x 130
energia del primer valor singular: 77.8%


### Validacion de K-SVD

La misma idea que con OMP: se genera un diccionario conocido, se
crean senales a partir de el, y se revisa si K-SVD lo recupera. La
medida es el coseno entre cada atomo verdadero y el aprendido mas
parecido. El valor absoluto porque un atomo invertido representa
igual de bien.

In [8]:
rng = np.random.default_rng(SEMILLA)
k_v, m_v, n_v, usados_v = 20, 50, 1500, 3

verdadero = normalizar_columnas(rng.standard_normal((k_v, m_v)))
sinteticas = normalizar_columnas(np.column_stack([
    verdadero[:, rng.choice(m_v, usados_v, replace=False)]
    @ rng.standard_normal(usados_v) for _ in range(n_v)]))

d_rec, _, hist_rec = ksvd(sinteticas, m=m_v, iteraciones=40,
                          error=1e-6, max_atomos=usados_v,
                          verboso=False)

parecido = np.abs(verdadero.T @ d_rec).max(axis=1)
err_rec = [h["error"] for h in hist_rec]

print(f"error {err_rec[0]:.4f} -> {err_rec[-1]:.4f}")
print(f"atomos con coseno > 0.99: {(parecido > 0.99).sum()}/{m_v}")
print(f"parecido promedio: {parecido.mean():.4f}")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Error por iteracion",
                                    "Parecido con el atomo verdadero"))
fig.add_trace(go.Scatter(y=err_rec, mode="lines",
                         line=dict(color=AZUL, width=2),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=parecido, nbinsx=25,
                           marker_color=AQUA, showlegend=False),
              row=1, col=2)
fig.update_xaxes(title_text="iteracion", row=1, col=1)
fig.update_xaxes(title_text="coseno", row=1, col=2)
fig.update_layout(height=360,
                  title="K-SVD recupera el diccionario que genero "
                        "los datos", **PLANTILLA)
fig.show()

error 0.4543 -> 0.1603
atomos con coseno > 0.99: 42/50
parecido promedio: 0.9763


El error no baja de forma estrictamente monotona, y eso **no es un
error de programacion**. OMP es voraz y no garantiza la solucion
optima, asi que de vez en cuando una iteracion sale peor que la
anterior.

## 5. Entrenamiento sobre el ECG real

Dos decisiones del montaje:

- **La particion es temporal, no aleatoria.** Se entrena con la
  primera mitad de las ventanas y se evalua con la segunda. Las
  ventanas contiguas de un ECG se parecen tanto entre si que
  repartirlas al azar dejaria en prueba vecinas directas de las de
  entrenamiento, lo que inflaria el resultado.
- **`D_0` sale de las propias senales**, no de ruido. Arrancar con
  columnas de `X` pone a K-SVD en la zona correcta del espacio desde
  el principio.

El entrenamiento tarda unos minutos, asi que si ya existe
`out/modelo.npz` se carga en vez de repetirlo.

In [9]:
import os

if os.path.exists(ARCHIVO_MODELO):
    datos = np.load(ARCHIVO_MODELO)
    d_ecg = datos["diccionario"]
    alpha_prueba = datos["alpha_prueba"]
    dispersion = datos["historia_dispersion"]
    print(f"cargado de {ARCHIVO_MODELO}")
else:
    d_ecg, alpha_entrena, historia = ksvd(entrena)
    alpha_prueba = omp_matriz(d_ecg, prueba)
    dispersion = np.array([h["dispersion"] for h in historia])

print(f"\nD de {d_ecg.shape[0]} x {d_ecg.shape[1]}")
print(f"error en prueba: {error_relativo(prueba, d_ecg @ alpha_prueba):.4f}")
print(f"atomos por senal: {dispersion_media(alpha_prueba):.2f}")

cargado de ../out/modelo.npz

D de 128 x 256
error en prueba: 0.1148
atomos por senal: 9.46


### Que mejora durante el entrenamiento

Aqui hay un detalle que confunde si no se dice: **el error de
reconstruccion no baja con las iteraciones**. Se queda clavado en
0.10 porque ese *es* el criterio de paro de OMP: en cuanto llega al
10% de error, deja de agregar atomos.

Lo que K-SVD mejora es la **dispersion**: cuantos atomos hacen falta
para alcanzar ese mismo error.

In [10]:
fig = go.Figure(go.Scatter(
    x=np.arange(1, len(dispersion) + 1), y=dispersion,
    mode="lines+markers", line=dict(color=AZUL, width=2),
    marker=dict(size=7), showlegend=False))
fig.update_layout(
    title="Atomos por senal a lo largo del entrenamiento",
    xaxis_title="iteracion de K-SVD",
    yaxis_title="atomos por senal", height=340, **PLANTILLA)
fig.show()

print(f"de {dispersion[0]:.2f} a {dispersion[-1]:.2f} atomos "
      f"para el mismo error")

de 10.62 a 8.42 atomos para el mismo error


## 6. Resultados

### Que aprendio el diccionario

Los atomos ordenados por uso, o sea por en cuantas senales participa
cada uno. Si el entrenamiento sirvio, aqui no hay ruido: hay
complejos QRS en distintas posiciones y polaridades.

In [11]:
usos_ecg = np.count_nonzero(alpha_prueba, axis=1)
orden = np.argsort(-usos_ecg)[:18]

fig = make_subplots(
    rows=3, cols=6,
    subplot_titles=[f"{usos_ecg[j]} senales" for j in orden],
    vertical_spacing=0.12, horizontal_spacing=0.03)

for i, j in enumerate(orden):
    fig.add_trace(go.Scatter(y=d_ecg[:, j], mode="lines",
                             line=dict(color=AZUL, width=1.5),
                             showlegend=False),
                  row=i // 6 + 1, col=i % 6 + 1)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_annotations(font_size=10)
fig.update_layout(height=520,
                  title="Los 18 atomos mas usados", **PLANTILLA)
fig.show()

### Original contra reconstruccion

El entregable que pide el enunciado. Los trazos van desplazados en
vertical porque encimados se tapan entre si.

In [12]:
col_p = elegir_ventana(prueba)
objetivo_p = prueba[:, col_p]
paso = (objetivo_p.max() - objetivo_p.min()) * 0.9
topes_p = (2, 4, 8, 16)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ms, y=objetivo_p, mode="lines",
                         name="original",
                         line=dict(color=TINTA, width=3)))

for i, tope in enumerate(topes_p, start=1):
    coef, _ = omp(d_ecg, objetivo_p, error=0.0, max_atomos=tope)
    recon = d_ecg @ coef
    err = error_relativo(objetivo_p, recon)
    fig.add_trace(go.Scatter(
        x=ms, y=recon - i * paso, mode="lines",
        name=f"{tope} atomos, error {err:.3f}",
        line=dict(width=2)))

fig.update_layout(
    title="Un latido y sus reconstrucciones",
    xaxis_title="ms", yaxis=dict(showticklabels=False),
    height=480, **PLANTILLA)
fig.show()

### Cuanto cuesta cada nivel de fidelidad

La curva de error contra el tope de atomos, medida sobre el conjunto
de prueba.

In [13]:
curva = curva_error(d_ecg, prueba[:, :400])
permitidos = [f["permitidos"] for f in curva]
errs = [f["error"] for f in curva]

fig = go.Figure(go.Scatter(
    x=permitidos, y=errs, mode="lines+markers",
    line=dict(color=AZUL, width=2), marker=dict(size=9),
    hovertemplate="%{x} atomos<br>error %{y:.4f}<extra></extra>",
    showlegend=False))
fig.update_layout(
    title="Error de reconstruccion segun el tope de atomos",
    xaxis_title="atomos permitidos por senal",
    yaxis_title="error relativo", height=380, **PLANTILLA)
fig.show()

for fila in curva:
    print(f"{fila['permitidos']:>3} atomos  ->  "
          f"error {fila['error']:.4f}")

  1 atomos  ->  error 0.3072
  2 atomos  ->  error 0.2180
  4 atomos  ->  error 0.1633
  6 atomos  ->  error 0.1384
  8 atomos  ->  error 0.1222
 10 atomos  ->  error 0.1116
 12 atomos  ->  error 0.1044
 16 atomos  ->  error 0.0950
 24 atomos  ->  error 0.0848
 32 atomos  ->  error 0.0778


## Conclusiones

**La implementacion funciona y esta verificada contra casos donde se
conoce la respuesta.** OMP recupera el soporte exacto en 99 de cada
100 senales sinteticas, y K-SVD reconstruye el diccionario que genero
los datos con coseno mayor a 0.99 en la mayoria de los atomos.

**Sobre el ECG, el diccionario aprende complejos QRS.** No es una
observacion cosmetica: significa que la base entrenada capturo la
estructura de la senal en lugar de memorizar ventanas.

**La compresion es real.** Una ventana de 128 muestras se reconstruye
al 10% de error con unos 9 atomos, y al 3% con 16. El costo de
guardar son los indices y los coeficientes, no las 128 muestras.

**Lo que mejora con el entrenamiento es la dispersion, no el error.**
Con un criterio de paro por error fijo, K-SVD no baja el error, baja
cuantos atomos hacen falta para alcanzarlo: de 10.62 a 8.42.

**La brecha entre entrenamiento y prueba es chica**, de 0.1010 a
0.1148, lo que indica que el diccionario generaliza y no memorizo las
ventanas con las que se entreno.
